## BGL LogAnomaly Anomaly Detection Experiment

This experiment focuses on training and evaluating the **LogAnomaly** model (LSTM + Multi-Head Attention) for **anomaly detection** on the BGL log dataset.

### I. Dataset and Source

* **Dataset:** BGL (Blue Gene/L Supercomputer) - LogHub preprocessed
* **Source Citation:** Adam Oliner and Jon Stearley (DSN 2007), Jieming Zhu et al. (ISSRE 2023)

### II. Model: LogAnomaly (LSTM + Attention)

* **Architecture:** **LSTM with Multi-Head Attention mechanism**
* **Key Features:**
    * Multi-Head Attention for long-range dependencies
    * Residual connections and Layer Normalization
    * Semantic embeddings support
* **Hyperparameters:**
    * `embedding_dim`: 384
    * `hidden_dim`: 128
    * `num_layers`: 2
    * `dropout`: 0.3
    * `n_heads`: 4
    * `use_attention`: True


### III. Task and Evaluation

* **Task:** **Next-Event Prediction** for Sequence-Based Anomaly Detection.
    * Anomalies detected using **Top-k Prediction** criteria.
    * `top_k`: 11
* **Evaluation Metrics:** Accuracy, Precision, Recall, F1 Score, and AUC.

### IV. Data Flow and Training

#### A. Preprocessing
* **Tokenization:** Log events are tokenized and mapped to **integer IDs** via a vocabulary.
* **Sequencing:** Log sequences are created using **fixed sliding windows** of size 20.
* **Data Split:**
    * Train: 70%
    * Validation: 15%
    * Test: 15%

#### B. Training
* **Batch Size:** 64
* **Learning Rate:** 0.001
* **Epochs:** Up to 50, with **Early Stopping** (`patience=10`).
* **Device:** Auto-detected (mps/cpu).

### V. Artifacts

* **Model Checkpoint:** `../../mdls/loganomaly_bgl_checkpoint.pt`
* **Results:** `../../results/bgl_loganomaly_results.json`
* **Visualizations:** `../../figures/loganomaly/`

### VI. Comparison with DeepLog

LogAnomaly extends DeepLog with:
1. **Multi-Head Attention** to capture long-range dependencies
2. **Residual connections** for better gradient flow
3. **Layer Normalization** for stable training
4. **Optional semantic embeddings** for better event representation

In [1]:
# Core imports
import numpy as np
import torch
import sys
from pathlib import Path
# Add project root to Python path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# Import modules
from src.data.dataset import create_data_loaders
from src.models.loganomaly import LogAnomalyModel
from src.engine.trainer import LogSeqTrainer
from src.utils.metrics import evaluate_model, print_metrics, save_experiment_results
from src.utils.data_loader import create_train_val_test_split, filter_normal_samples, load_loghub
from src.utils.visualizer import UniversalAnomalyVisualizer
from src.utils.seed import seed_everything

### 1. Setup and Imports

In [ ]:
# Define paths
DATA_DIR = '../../data/bgl/preprocessed'
seed_everything(42)

### 2. Load and Prepare Data

In [3]:
X, y, vocab = load_loghub(DATA_DIR)

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Events: {sorted(vocab.keys())[:10]}")

INFO:src.utils.data_loader:Loading data from LogHub preprocessing: ../../data/bgl/preprocessed
INFO:src.utils.data_loader:Loaded data:
INFO:src.utils.data_loader:  - Sequences: (949589, 20)
INFO:src.utils.data_loader:  - Labels: (949589,)
INFO:src.utils.data_loader:  - Normal: 868556, Anomaly: 81033
INFO:src.utils.data_loader:  - Loaded vocabulary: 371 events


Vocab size: 371
Events: ['<PAD>', '<UNK>', 'E1', 'E10', 'E100', 'E101', 'E102', 'E103', 'E1035', 'E1036']


In [4]:
# Split data (70/15/15)
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']

# Filter to keep only normal samples for semi-supervised training
X_train, y_train = filter_normal_samples(X_train, y_train, verbose=True)

INFO:src.utils.data_loader:Splitting data: train=0.7, val=0.15, test=0.15
INFO:src.utils.data_loader:Split complete:
INFO:src.utils.data_loader:  - Train: 664711 samples (56723 anomalies)
INFO:src.utils.data_loader:  - Val:   142439 samples (12155 anomalies)
INFO:src.utils.data_loader:  - Test:  142439 samples (12155 anomalies)
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:FILTERING TRAINING DATA FOR SEMI-SUPERVISED LEARNING
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:Original training size: 664711 samples
INFO:src.utils.data_loader:  Normal samples: 607,988 (91.47%)
INFO:src.utils.data_loader:  Anomaly samples: 56,723 (8.53%)
INFO:src.utils.data_loader:
Filtered training size: 607,988 samples (NORMAL ONLY)
INFO:src.utils.data_loader:Removed 56,723 anomalies from training set
INFO:src.utils.data_loader:✓ Training data is now pur

In [5]:
# Get optimal dataloader kwargs
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
loader_kwargs = LogSeqTrainer.get_optimal_dataloader_kwargs(device)
loader_kwargs

{'pin_memory': False, 'num_workers': 4}

In [6]:
# Create data loaders
batch_size = 64
train_loader, val_loader, test_loader = create_data_loaders(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    batch_size=batch_size,
    **loader_kwargs
)

In [7]:
#  Load the pre-computed embeddings
semantic_vectors = torch.load(f'{DATA_DIR}/semantic_embeddings.pt')
emb_dim = semantic_vectors.shape[1] 
print(f"Loaded semantic embeddings with dim: {emb_dim}")

Loaded semantic embeddings with dim: 384


### 3. Create LogAnomaly Model

LogAnomaly enhances DeepLog with:
- **Multi-Head Attention** (4 heads)
- **Residual connections**
- **Layer Normalization**
- **Semantic Embeddings** (pre-trained from log templates)

In [8]:
# Create LogAnomaly model
model = LogAnomalyModel(
    vocab_size=vocab_size,
    embedding_dim=emb_dim,  
    hidden_dim=128,
    num_layers=2,
    dropout=0.3,
    use_attention=True, 
    n_heads=4, 
    semantic_embeddings=semantic_vectors 
)

In [9]:
# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Embedding dimension: {emb_dim}")
print(f"  Attention enabled: {model.use_attention}")
print(f"  Semantic embeddings: {'✓ Loaded' if semantic_vectors is not None else '✗ Random init'}")


Model Architecture:
  Total parameters: 651,891
  Trainable parameters: 651,891
  Embedding dimension: 384
  Attention enabled: True
  Semantic embeddings: ✓ Loaded


In [10]:
# Train
learning_rate = 0.001
patience = 10

print(f"Training on device: {device}")

trainer = LogSeqTrainer(model, device=device, learning_rate=learning_rate)

history = trainer.fit(
    train_loader, val_loader,
    num_epochs=50,
    early_stopping_patience=patience,
    print_every=5  # Print every 5 epochs
)

Training on device: mps


Training: 100%|██████████| 9500/9500 [00:53<00:00, 177.58it/s]



Epoch 1/50 - 64.35s
  Train Loss: 0.1964
  Val Loss:   0.8225
  ✓ New best model (val_loss: 0.8225)


Training: 100%|██████████| 9500/9500 [00:53<00:00, 176.44it/s]



Epoch 5/50 - 62.84s
  Train Loss: 0.1636
  Val Loss:   0.8057
  No improvement (1/10)


...


Epoch 20/50 - 61.44s
  Train Loss: 0.1615
  Val Loss:   0.7721
  No improvement (3/10)


Training: 100%|██████████| 9500/9500 [00:53<00:00, 178.74it/s]



Epoch 25/50 - 62.13s
  Train Loss: 0.1613
  Val Loss:   0.7521
  No improvement (8/10)


Training: 100%|██████████| 9500/9500 [00:53<00:00, 178.71it/s]



Early stopping triggered after 27 epochs

✓ Loaded best model (val_loss: 0.7324)
Total training time: 1682.12s


### 4. Train LogAnomaly

In [16]:
# Evaluate
# Capture predictions, true labels, AND anomaly scores from the loader
top_k = 11
predictions, true_labels, anomaly_scores = trainer.detect_anomalies(
    test_loader,
    top_k=top_k,
    return_scores=True
)

metrics = evaluate_model(predictions, true_labels) 
print_metrics(metrics)

Detecting anomalies: 100%|██████████| 2226/2226 [00:15<00:00, 139.40it/s]



EVALUATION METRICS
Accuracy:  0.9889 (98.89%)
Precision: 0.8890
Recall:    0.9936
F1-Score:  0.9384

Confusion Matrix:
              Predicted
              Normal  Anomaly
Actual Normal   128776     1508
       Anomaly      78    12077


### 5. Evaluation

Evaluate using top-k prediction

In [17]:
# # Save experiment results
save_experiment_results(
    filepath="../../results/bgl_loganomaly_results.json",
    dataset="BGL",
    model_name="LogAnomaly",
    device=device,
    model=model,
    history=history,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    max_len=int(np.percentile([len(s) for s in X_train], 95)),
    metrics=metrics,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    top_k=top_k,
    # LogAnomaly-specific params
    use_attention=True,
    n_heads=4
)

✓ Results saved to ../../results/bgl_loganomaly_results.json


{'dataset': 'BGL',
 'model': 'LogAnomaly',
 'device': 'mps',
 'architecture': {'vocab_size': 371,
  'embedding_dim': 384,
  'hidden_dim': 128,
  'num_layers': 2,
  'dropout': 0.3,
  'use_attention': True,
  'num_attention_heads': 4},
 'training': {'num_epochs': 27,
  'batch_size': 64,
  'learning_rate': 0.001,
  'early_stopping_patience': 10,
  'best_val_loss': 0.7324049350573796,
  'training_time_seconds': 1682.119311094284},
 'data': {'train_size': 607988,
  'val_size': 142439,
  'test_size': 142439,
  'max_sequence_length': 20,
  'train_normal_only': True},
 'detection': {'top_k': 11, 'method': 'next_event_prediction'},
 'metrics': {'accuracy': 0.988865409052296,
  'precision': 0.8889952153110048,
  'recall': 0.9935828877005347,
  'f1': 0.9383838383838384,
  'confusion_matrix': [[128776, 1508], [78, 12077]]}}

### 7. Save Results and Model

In [ ]:
# Initialize visualizer with save directory
viz = UniversalAnomalyVisualizer(model_name="LogAnomaly", save_dir="../../figures/loganomaly")

# Plot 1: Training History (Loss curves)
viz.plot_training_history(history)

# Plot 2: Confusion Matrix
viz.plot_confusion_matrix(true_labels, predictions, title_suffix="(BGL Test Set)")

# Plot 3: Anomaly Score Distribution
viz.plot_anomaly_score_distribution(true_labels, anomaly_scores, bins=50)

# Plot 4: ROC Curve
viz.plot_roc_curve(true_labels, anomaly_scores)

# Plot 5: Precision-Recall Curve
viz.plot_precision_recall_curve(true_labels, anomaly_scores)

print("\n✓ All visualizations generated and saved using UniversalAnomalyVisualizer")

### 6. Visualizations

Generate comprehensive visualizations including:
1. Training history
2. Confusion matrix
3. Anomaly score distribution
4. ROC curve
5. Precision-Recall curve

In [18]:
save_dir = Path('../../mdls')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'loganomaly_bgl_checkpoint.pt'
trainer.save_model(str(save_path))

✓ Model saved to ../../mdls/loganomaly_bgl_checkpoint.pt
